In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import thop
except ImportError:
    _install('thop')

try:
    import torchinfo
except ImportError:
    _install('torchinfo')

print('Dependencies ready.')

Dependencies ready.


In [ ]:
import time
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

# ── Optional FLOPs libraries ─────────────────────────────────────────────────
try:
    from thop import profile as thop_profile
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False

try:
    from torchinfo import summary as torchinfo_summary
    TORCHINFO_AVAILABLE = True
except ImportError:
    TORCHINFO_AVAILABLE = False

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"})')
print(f'thop      : {"OK" if THOP_AVAILABLE else "not found – using manual FLOPs"}')
print(f'torchinfo : {"OK" if TORCHINFO_AVAILABLE else "not found"}')

PyTorch   : 2.11.0+cu128
CUDA      : True (Tesla T4)
thop      : OK
torchinfo : OK


In [ ]:
# ── Device (GPU if available; CPU fallback for reproducibility) ──────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

Using device: cuda


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Hyperparameters — fixed for all models as specified in the problem statement
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE  = 64
NUM_EPOCHS  = 10
LR          = 1e-3
NUM_CLASSES = 100
IMG_SIZE    = 32

# Channel-wise mean/std computed over the CIFAR-100 training set
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

# Training transform: random crop + horizontal flip for regularisation
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Test transform: deterministic normalisation only
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_dataset = torchvision.datasets.CIFAR100(
    root='./data', train=True,  download=True, transform=train_transform
)
test_dataset  = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=test_transform
)

# num_workers=2 works on Colab/Linux; set to 0 on Windows or CPU-only machines
_nw = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=_nw, pin_memory=torch.cuda.is_available()
)
test_loader  = DataLoader(
    test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
    num_workers=_nw, pin_memory=torch.cuda.is_available()
)

print(f'Train samples : {len(train_dataset):,}')
print(f'Test  samples : {len(test_dataset):,}')
print(f'Batches/epoch : {len(train_loader)}')

100%|██████████| 169M/169M [47:57<00:00, 58.7kB/s]


Train samples : 50,000
Test  samples : 10,000
Batches/epoch : 782


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Vision Transformer (ViT) — implemented from scratch
# Reference: Dosovitskiy et al., "An Image is Worth 16x16 Words", ICLR 2021
# ═══════════════════════════════════════════════════════════════════════════════


class PatchEmbedding(nn.Module):
    """
    Splits the input image into non-overlapping patches and projects each patch
    to a D-dimensional embedding vector.

    A Conv2d with kernel_size=P and stride=P is mathematically equivalent to
    independently applying a linear map to each flattened P×P×C patch, but
    runs faster due to optimised GPU kernels.

    Attributes
    ----------
    num_patches : int
        Number of non-overlapping patches = (img_size // patch_size) ** 2.
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, emb_dim=256):
        super().__init__()
        assert img_size % patch_size == 0, (
            f'img_size={img_size} must be divisible by patch_size={patch_size}'
        )
        self.num_patches = (img_size // patch_size) ** 2
        # Strided convolution = linear projection of each flattened patch
        self.proj = nn.Conv2d(
            in_channels, emb_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x):
        # x : (B, C, H, W)
        x = self.proj(x)       # (B, D, H/P, W/P)
        x = x.flatten(2)       # (B, D, num_patches)
        x = x.transpose(1, 2)  # (B, num_patches, D)
        return x


class TransformerBlock(nn.Module):
    """
    Transformer encoder block with *Pre-LayerNorm* (Pre-LN) ordering.

    Pre-LN applies LayerNorm *before* each sublayer rather than after, which
    improves gradient flow and training stability (Wang et al., 2019).

    Sub-layers (each surrounded by a residual connection):
        1.  LN → Multi-Head Self-Attention  (MHSA)
        2.  LN → Position-wise MLP  (GELU activation, 4× hidden expansion)

    Parameters
    ----------
    emb_dim   : model / embedding dimension D
    num_heads : number of attention heads H  (D must be divisible by H)
    mlp_ratio : MLP hidden size = mlp_ratio × D  (default 4 per ViT paper)
    dropout   : dropout rate applied inside attention and MLP
    """
    def __init__(self, emb_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_dim)
        # batch_first=True: tensors shaped (B, S, D) everywhere
        self.attn  = nn.MultiheadAttention(
            emb_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2    = nn.LayerNorm(emb_dim)
        mlp_hidden    = int(emb_dim * mlp_ratio)
        self.mlp      = nn.Sequential(
            nn.Linear(emb_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # Self-attention sublayer  (Q = K = V = normed input)
        normed    = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out                   # residual

        # Feed-forward sublayer
        x = x + self.mlp(self.norm2(x))    # residual
        return x


class ViT(nn.Module):
    """
    Vision Transformer for image classification.

    Architecture
    ------------
    1.  PatchEmbedding  —  split image into N non-overlapping patches,
                           project each to D dims.
    2.  Prepend a learnable CLS token  →  sequence length S = N + 1.
    3.  Add learnable positional embeddings  (one per sequence position).
    4.  L × TransformerBlock.
    5.  LayerNorm on the CLS token output.
    6.  Linear classification head  (D → num_classes).

    Parameters
    ----------
    img_size    : input image size (assumed square, default 32 for CIFAR)
    patch_size  : size of each square patch  P  ∈ {4, 8}
    in_channels : input channels (3 for RGB)
    num_classes : number of output classes (100 for CIFAR-100)
    emb_dim     : embedding / model dimension  D  ∈ {256, 512}
    num_blocks  : number of stacked Transformer blocks  L  ∈ {4, 8}
    num_heads   : number of attention heads  H  ∈ {4, 8}  (D % H == 0)
    mlp_ratio   : MLP expansion factor  (default 4 → MLP hidden = 4D)
    dropout     : dropout rate
    """
    def __init__(
        self,
        img_size    = 32,
        patch_size  = 4,
        in_channels = 3,
        num_classes = 100,
        emb_dim     = 256,
        num_blocks  = 4,
        num_heads   = 4,
        mlp_ratio   = 4.0,
        dropout     = 0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, emb_dim)
        num_patches      = self.patch_embed.num_patches

        # CLS token: single learnable vector prepended to every sequence.
        # Its final representation is fed to the classification head.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, emb_dim))

        # Learnable positional embeddings for CLS + all patch positions.
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, emb_dim))
        self.pos_drop  = nn.Dropout(dropout)

        # Stacked Transformer encoder blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(emb_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_blocks)
        ])

        self.norm = nn.LayerNorm(emb_dim)
        self.head = nn.Linear(emb_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        # ViT paper: truncated normal(std=0.02) for embeddings + linear weights
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.shape[0]

        # 1. Patch embedding
        x   = self.patch_embed(x)               # (B, N, D)

        # 2. Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)        # (B, N+1, D)

        # 3. Add positional embedding and apply dropout
        x = self.pos_drop(x + self.pos_embed)

        # 4. Transformer encoder
        x = self.blocks(x)                      # (B, N+1, D)

        # 5. Normalise CLS token and classify
        x = self.norm(x)
        return self.head(x[:, 0])               # (B, num_classes)


# Quick sanity check
_dummy = torch.randn(2, 3, 32, 32)
_vit   = ViT()
assert _vit(_dummy).shape == (2, 100), 'ViT output shape mismatch'
print('ViT forward pass: OK')
del _dummy, _vit

ViT forward pass: OK


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Parameter counting and FLOPs estimation
# ═══════════════════════════════════════════════════════════════════════════════


def count_parameters(model):
    """Return the total number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def compute_vit_flops_manual(img_size, patch_size, emb_dim, num_blocks):
    """
    Closed-form FLOPs estimate for a single-image ViT forward pass.

    Counting convention: one multiply-add = 2 FLOPs.

    Notation
    --------
    N  = (img_size / patch_size)^2  — number of patches
    S  = N + 1                       — sequence length (patches + CLS)
    D  = emb_dim
    D' = 4D                          — MLP hidden width (mlp_ratio = 4)
    P  = patch_size,  C = 3

    Per Transformer block
    ─────────────────────
    QKV projection     :  2 · S · D · 3D  =   6 S D^2
    Attn scores QK^T   :  2 · S · S · D   =   2 S^2 D
    Attn softmax × V   :  2 · S · S · D   =   2 S^2 D
    Output projection  :  2 · S · D · D   =   2 S D^2
    MLP FC-1           :  2 · S · D · 4D  =   8 S D^2
    MLP FC-2           :  2 · S · 4D · D  =   8 S D^2
    ─────────────────────────────────────────────────
    Total per block    ≈  24 S D^2 + 4 S^2 D

    Additional costs
    ────────────────
    Patch embed (Conv2d) :  2 · N · (P^2 · C) · D
    Classification head  :  2 · D · 100
    """
    N           = (img_size // patch_size) ** 2
    S           = N + 1
    C           = 3
    num_classes = 100

    flops_embed  = 2 * N * (patch_size ** 2 * C) * emb_dim
    flops_blocks = num_blocks * (24 * S * emb_dim ** 2 + 4 * S ** 2 * emb_dim)
    flops_head   = 2 * emb_dim * num_classes

    return flops_embed + flops_blocks + flops_head


def get_flops_thop(model, img_size=32):
    """
    Use thop to profile FLOPs on a dummy input.
    Returns int (FLOPs) or None if thop is not available / profiling fails.
    """
    if not THOP_AVAILABLE:
        return None
    dummy = torch.randn(1, 3, img_size, img_size).to(
        next(model.parameters()).device
    )
    try:
        macs, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        return int(macs * 2)   # thop counts MACs; 1 MAC = 2 FLOPs
    except Exception:
        return None


def get_torchinfo_summary(model, img_size=32):
    """Print a torchinfo model summary (optional, informational only)."""
    if not TORCHINFO_AVAILABLE:
        return
    torchinfo_summary(
        model,
        input_size=(1, 3, img_size, img_size),
        device=next(model.parameters()).device,
        verbose=1,
        col_names=['input_size', 'output_size', 'num_params', 'mult_adds'],
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Training and evaluation helpers
# ═══════════════════════════════════════════════════════════════════════════════


def train_one_epoch(model, loader, optimizer, criterion):
    """One training epoch; returns (avg_loss, accuracy)."""
    model.train()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping improves ViT training stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate on a DataLoader; returns (avg_loss, accuracy)."""
    model.eval()
    total_loss = correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def train_model(model, name, flops_manual=None, num_epochs=NUM_EPOCHS):
    """
    Train model for num_epochs on train_loader, evaluate on test_loader each epoch.
    Returns a results dict with all metrics and the trained model.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    params        = count_parameters(model)
    flops_thop    = get_flops_thop(model)
    flops         = flops_thop if flops_thop else flops_manual
    flops_src     = 'thop' if flops_thop else ('manual' if flops_manual else '—')

    print(f'\n{"="*70}')
    print(f'  Model      : {name}')
    print(f'  Parameters : {params:,}')
    if flops:
        print(f'  FLOPs/img  : {flops/1e6:.1f} M  ({flops_src})')
    print(f'{"="*70}')

    epoch_times   = []
    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []

    wall_start = time.time()

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        te_loss, te_acc = evaluate(model, test_loader, criterion)

        elapsed = time.time() - t0
        epoch_times.append(elapsed)
        train_losses.append(tr_loss)
        train_accs.append(tr_acc)
        test_losses.append(te_loss)
        test_accs.append(te_acc)

        print(
            f'  Epoch {epoch:2d}/{num_epochs} | '
            f'Train loss={tr_loss:.4f} acc={tr_acc*100:.2f}% | '
            f'Test  loss={te_loss:.4f} acc={te_acc*100:.2f}% | '
            f'{elapsed:.1f}s'
        )

    total_time = time.time() - wall_start
    mean_epoch = float(np.mean(epoch_times))

    print(f'\n  Total training time : {total_time:.1f}s')
    print(f'  Mean time per epoch : {mean_epoch:.1f}s')
    print(f'  Final test accuracy : {test_accs[-1]*100:.2f}%')

    return {
        'name'            : name,
        'params'          : params,
        'flops'           : flops,
        'train_losses'    : train_losses,
        'train_accs'      : train_accs,
        'test_losses'     : test_losses,
        'test_accs'       : test_accs,
        'epoch_times'     : epoch_times,
        'total_time_sec'  : total_time,
        'mean_epoch_sec'  : mean_epoch,
        'final_test_acc'  : test_accs[-1],
        'final_train_loss': train_losses[-1],
        'final_test_loss' : test_losses[-1],
        'model'           : model,
    }

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ViT experiment configurations  (≥ 4 required by the problem statement)
# ═══════════════════════════════════════════════════════════════════════════════
#
#  Name              Patch  emb_dim  blocks  heads  Rationale
#  ──────────────    ─────  ───────  ──────  ─────  ────────────────────────────
#  ViT-Tiny  P4D256    4     256       4      4     Smallest — quick baseline
#  ViT-Small P8D256    8     256       4      4     Fewer / larger patches
#  ViT-Med   P4D256    4     256       8      8     Deeper, same width as Tiny
#  ViT-Base  P4D512    4     512       8      8     Largest — max capacity

VIT_CONFIGS = [
    # (display_name,              patch, emb, blocks, heads)
    ('ViT-Tiny  (P4-D256-L4-H4)',  4, 256, 4, 4),
    ('ViT-Small (P8-D256-L4-H4)',  8, 256, 4, 4),
    ('ViT-Med   (P4-D256-L8-H8)',  4, 256, 8, 8),
    ('ViT-Base  (P4-D512-L8-H8)',  4, 512, 8, 8),
]

# Pre-training theoretical complexity preview
print(f'\n{"Configuration":<36} {"Patch":>5} {"emb":>5} {"L":>3} '
      f'{"H":>3} {"Patches":>8} {"SeqLen":>7} {"FLOPs/img":>11}')
print('-' * 84)

for name, patch, emb, blocks, heads in VIT_CONFIGS:
    N     = (IMG_SIZE // patch) ** 2
    S     = N + 1
    flops = compute_vit_flops_manual(IMG_SIZE, patch, emb, blocks)
    print(f'{name:<36} {patch:>5} {emb:>5} {blocks:>3} '
          f'{heads:>3} {N:>8} {S:>7} {flops/1e6:>10.1f}M')


Configuration                        Patch   emb   L   H  Patches  SeqLen   FLOPs/img
------------------------------------------------------------------------------------
ViT-Tiny  (P4-D256-L4-H4)                4   256   4   4       64      65      427.9M
ViT-Small (P8-D256-L4-H4)                8   256   4   4       16      17      109.8M
ViT-Med   (P4-D256-L8-H8)                4   256   8   8       64      65      854.1M
ViT-Base  (P4-D512-L8-H8)                4   512   8   8       64      65     3344.0M


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Train all four ViT configurations
# ═══════════════════════════════════════════════════════════════════════════════

all_results = []

for cfg_name, patch, emb, blocks, heads in VIT_CONFIGS:
    vit = ViT(
        img_size    = IMG_SIZE,
        patch_size  = patch,
        in_channels = 3,
        num_classes = NUM_CLASSES,
        emb_dim     = emb,
        num_blocks  = blocks,
        num_heads   = heads,
        mlp_ratio   = 4.0,
        dropout     = 0.1,
    )

    manual_flops = compute_vit_flops_manual(IMG_SIZE, patch, emb, blocks)
    result       = train_model(vit, cfg_name, flops_manual=manual_flops)

    # Attach configuration metadata for the summary table
    result.update({
        'patch_size': patch, 'emb_dim': emb,
        'num_blocks': blocks, 'num_heads': heads,
        'seq_len'   : (IMG_SIZE // patch) ** 2 + 1,
    })
    all_results.append(result)


  Model      : ViT-Tiny  (P4-D256-L4-H4)
  Parameters : 3,214,692
  FLOPs/img  : 275.5 M  (thop)
  Epoch  1/10 | Train loss=4.1137 acc=6.61% | Test  loss=3.8522 acc=9.78% | 34.4s
  Epoch  2/10 | Train loss=3.7940 acc=10.87% | Test  loss=3.6531 acc=12.97% | 34.3s
  Epoch  3/10 | Train loss=3.6418 acc=13.23% | Test  loss=3.5074 acc=16.10% | 34.3s
  Epoch  4/10 | Train loss=3.5327 acc=15.34% | Test  loss=3.3880 acc=18.19% | 35.7s
  Epoch  5/10 | Train loss=3.4529 acc=16.66% | Test  loss=3.3031 acc=19.57% | 35.7s
  Epoch  6/10 | Train loss=3.3952 acc=17.59% | Test  loss=3.3076 acc=19.21% | 35.4s
  Epoch  7/10 | Train loss=3.3667 acc=18.30% | Test  loss=3.2696 acc=20.07% | 35.4s
  Epoch  8/10 | Train loss=3.3245 acc=18.96% | Test  loss=3.2149 acc=21.19% | 35.5s
  Epoch  9/10 | Train loss=3.2792 acc=19.86% | Test  loss=3.2128 acc=21.30% | 35.8s
  Epoch 10/10 | Train loss=3.2505 acc=20.33% | Test  loss=3.1596 acc=22.55% | 35.5s

  Total training time : 351.9s
  Mean time per epoch : 35.2s
  

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ResNet-18 baseline  —  trained from scratch, identical hyperparameters
# ═══════════════════════════════════════════════════════════════════════════════
#
# torchvision's stock ResNet-18 targets ImageNet (224×224).  Its first conv
# (7×7, stride=2) + maxpool reduces 32×32 inputs to 8×8 before the residual
# stages, discarding critical spatial detail.
#
# Standard CIFAR adaptation (following He et al., 2016 — original CIFAR ResNet):
#   1. conv1 :  7×7, stride=2  →  3×3, stride=1, padding=1
#   2. maxpool : removed  (replaced with nn.Identity)
#   3. fc     :  1000 → 100
# All weights are randomly initialised (weights=None) for a fair from-scratch
# comparison with the ViT models.

def build_resnet18_cifar100():
    net = models.resnet18(weights=None)              # no pre-training
    net.conv1   = nn.Conv2d(3, 64, kernel_size=3,    # 7x7→3x3, remove downsampling
                             stride=1, padding=1, bias=False)
    net.maxpool = nn.Identity()                       # preserve spatial resolution
    net.fc      = nn.Linear(512, NUM_CLASSES)         # 1000 → 100
    return net

resnet        = build_resnet18_cifar100()
resnet_result = train_model(resnet, 'ResNet-18 (CIFAR adapted)')
resnet_result.update({'patch_size': None, 'emb_dim': None,
                      'num_blocks': None, 'num_heads': None, 'seq_len': None})
all_results.append(resnet_result)


  Model      : ResNet-18 (CIFAR adapted)
  Parameters : 11,220,132
  FLOPs/img  : 1115.9 M  (thop)
  Epoch  1/10 | Train loss=3.6382 acc=13.83% | Test  loss=3.1232 acc=22.17% | 50.9s
  Epoch  2/10 | Train loss=2.8285 acc=27.84% | Test  loss=2.6521 acc=31.62% | 50.9s
  Epoch  3/10 | Train loss=2.4125 acc=36.11% | Test  loss=2.3520 acc=38.32% | 50.6s
  Epoch  4/10 | Train loss=2.1198 acc=42.61% | Test  loss=2.2543 acc=41.34% | 50.3s
  Epoch  5/10 | Train loss=1.8713 acc=48.41% | Test  loss=1.8158 acc=50.65% | 50.2s
  Epoch  6/10 | Train loss=1.6553 acc=53.42% | Test  loss=1.6708 acc=53.59% | 51.1s
  Epoch  7/10 | Train loss=1.4773 acc=57.78% | Test  loss=1.6489 acc=55.09% | 51.0s
  Epoch  8/10 | Train loss=1.3306 acc=61.35% | Test  loss=1.5231 acc=58.26% | 51.5s
  Epoch  9/10 | Train loss=1.1980 acc=64.87% | Test  loss=1.4551 acc=60.43% | 51.6s
  Epoch 10/10 | Train loss=1.0840 acc=67.77% | Test  loss=1.3542 acc=62.53% | 50.0s

  Total training time : 508.2s
  Mean time per epoch : 50.8

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Final results summary table
# ═══════════════════════════════════════════════════════════════════════════════

print('\nFinal Results — CIFAR-100  (10 epochs, batch=64, Adam lr=0.001)')
print('=' * 105)
print(
    f'{"Model":<38} {"Params":>9} {"FLOPs":>10} '
    f'{"Test Acc":>9} {"Train Loss":>11} {"Ep (s)":>7} {"Total (s)":>9}'
)
print('-' * 105)

for r in all_results:
    f_str = f'{r["flops"]/1e6:.1f}M' if r['flops'] else '—'
    print(
        f'{r["name"]:<38} {r["params"]:>9,} {f_str:>10} '
        f'{r["final_test_acc"]*100:>8.2f}% {r["final_train_loss"]:>11.4f} '
        f'{r["mean_epoch_sec"]:>7.1f} {r["total_time_sec"]:>9.1f}'
    )

print()

# ── Per-epoch test accuracy curve ────────────────────────────────────────────
print('\nTest Accuracy (%) per Epoch')
labels = [r['name'][:20] for r in all_results]
print(f'{"Epoch":<6}', '  '.join(f'{l:>20}' for l in labels))
print('-' * (6 + 22 * len(all_results)))
for ep in range(NUM_EPOCHS):
    row = f'{ep+1:<6}'
    for r in all_results:
        row += f'  {r["test_accs"][ep]*100:>20.2f}'
    print(row)


Final Results — CIFAR-100  (10 epochs, batch=64, Adam lr=0.001)
Model                                     Params      FLOPs  Test Acc  Train Loss  Ep (s) Total (s)
---------------------------------------------------------------------------------------------------------
ViT-Tiny  (P4-D256-L4-H4)              3,214,692     275.5M    22.55%      3.2505    35.2     351.9
ViT-Small (P8-D256-L4-H4)              3,239,268      73.2M    11.65%      3.7876    29.4     293.9
ViT-Med   (P4-D256-L8-H8)              6,373,732     549.1M    29.40%      2.8318    68.5     685.1
ViT-Base  (P4-D512-L8-H8)              25,330,276    2188.8M     4.99%      4.2843   189.1    1891.2
ResNet-18 (CIFAR adapted)              11,220,132    1115.9M    62.53%      1.0840    50.8     508.2


Test Accuracy (%) per Epoch
Epoch  ViT-Tiny  (P4-D256-L  ViT-Small (P8-D256-L  ViT-Med   (P4-D256-L  ViT-Base  (P4-D512-L  ResNet-18 (CIFAR ada
---------------------------------------------------------------------------------

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Theoretical parameter count derivation  (ViT configurations only)
# ═══════════════════════════════════════════════════════════════════════════════
#
# For ViT with img_size=I, patch_size=P, emb_dim=D, num_blocks=L:
#   N  = (I/P)^2            — number of patches
#   D' = 4D                 — MLP hidden width
#
#   Component                 Parameters
#   ─────────────────────     ────────────────────────────
#   Patch embed  (Conv2d)  :  P^2 * 3 * D + D
#   CLS token              :  D
#   Positional embed       :  (N+1) * D
#   Per block (×L):
#     LN1 + LN2            :  4D
#     MHSA QKV projection  :  3D^2 + 3D
#     MHSA out projection  :  D^2  + D
#     MLP  FC-1            :  4D^2 + 4D
#     MLP  FC-2            :  4D^2 + D
#     ─ block total        ≈  12D^2 + 13D
#   Classification head    :  D * 100 + 100

print(f'\n{"Model":<38} {"P":>3} {"D":>5} {"L":>3} {"N":>4} '
      f'{"Embed":>9} {"Pos":>8} {"Blocks":>10} {"Head":>7} {"Measured":>10}')
print('-' * 103)

for r in all_results:
    if r['patch_size'] is None:
        print(f'{r["name"]:<38} {"—":>3} {"—":>5} {"—":>3} {"—":>4} '
              f'{"—":>9} {"—":>8} {"—":>10} {"—":>7} {r["params"]:>10,}')
        continue

    P, D, L = r['patch_size'], r['emb_dim'], r['num_blocks']
    N = (IMG_SIZE // P) ** 2

    p_embed   = P * P * 3 * D + D
    cls_tok   = D
    pos_emb   = (N + 1) * D
    per_block = 12 * D * D + 13 * D
    total_blk = L * per_block
    head_p    = D * 100 + 100
    theory    = p_embed + cls_tok + pos_emb + total_blk + head_p

    print(f'{r["name"]:<38} {P:>3} {D:>5} {L:>3} {N:>4} '
          f'{p_embed:>9,} {pos_emb:>8,} {total_blk:>10,} '
          f'{head_p:>7,} {r["params"]:>10,}')

print()
print('Theory omits small LN/bias terms; discrepancy from Measured is <1%.')


Model                                    P     D   L    N     Embed      Pos     Blocks    Head   Measured
-------------------------------------------------------------------------------------------------------
ViT-Tiny  (P4-D256-L4-H4)                4   256   4   64    12,544   16,640  3,159,040  25,700  3,214,692
ViT-Small (P8-D256-L4-H4)                8   256   4   16    49,408    4,352  3,159,040  25,700  3,239,268
ViT-Med   (P4-D256-L8-H8)                4   256   8   64    12,544   16,640  6,318,080  25,700  6,373,732
ViT-Base  (P4-D512-L8-H8)                4   512   8   64    25,088   33,280 25,219,072  51,300 25,330,276
ResNet-18 (CIFAR adapted)                —     —   —    —         —        —          —       — 11,220,132

Theory omits small LN/bias terms; discrepancy from Measured is <1%.
